In [2]:
# 一、准备文档
from langchain_text_splitters import MarkdownHeaderTextSplitter


def prepare_docs():
    # =============1.准备文档================
    with open("../day07/resources/评估.md", 'r', encoding='utf-8') as f:
        doc = f.read()

    # =============2.切分文档================
    # 切分依据，这里是按照三级标题
    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]

    # 创建切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on,
        strip_headers=False  # 在正文中是否去除标题，False，不去除
    )

    # 切分文档, 返回的是 list[Document]
    docs = markdown_splitter.split_text(doc)

    # 为了方便查看，给每个Document加一个id，从1开始递增，但要注意的是LangChain的文档id要求是str
    for i, doc in enumerate(docs, start=1):
        doc.id = str(i)
    return docs

docs = prepare_docs()

In [5]:
# 二、初始化向量模型
from langchain_ollama import OllamaEmbeddings

# 1.初始化向量模型
embaddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",
    dimensions=1024
)

In [6]:
from app.core.config import settings
from langchain_milvus import Milvus, BM25BuiltInFunction

vector_store = Milvus(
    embedding_function=embaddings,                   # 稠密向量生成模型
    collection_name="langchain_connection",          # 集合名称
    connection_args={"uri":settings.rag.milvus_url}, # 设置服务端地址,注意：URI
    drop_old=True,                                   # 删除旧的集合，避免重复创建
    vector_field=["dense", "sparse"],                # 自定义向量字段名称，必须是[稠密向量字段名称,稀疏向量字段名称]
    builtin_function=BM25BuiltInFunction(            # 生成稀疏向量的算法
        input_field_names="text",                    # 指定从哪个字段生成稀疏向量
        output_field_names="sparse",                 # 指定生成的稀疏向量写入到哪个字段
        analyzer_params={"type":"chinese"},          # 指定中文分词器
        function_name="text_2_sparse"                # 指定函数名称，保证名称不重复即可
    )
)

In [ ]:
# 添加数据
res = vector_store.add_documents(docs, ids=[doc.id for doc in docs])
print(res)

In [ ]:
# 1.删除，传入id集合
vector_store.delete(["13", "14"])

In [ ]:
from langchain_core.documents import Document
# 加权融合混合检索
doc_list: list[tuple[Document, float]] =(vector_store
                                         .similarity_search_with_score(
    query="给教师的建议是谁写的", ranker_type="weighted", ranker_params={"weights": [0.6, 0.4]}
))

for doc, score in doc_list:
    print(f"==========文档评分:{score}==========")
    print(doc.model_dump_json(indent=2))

In [ ]:
from langchain_core.documents import Document
# RRF融合混合检索
doc_list: list[tuple[Document, float]] =(vector_store
                                         .similarity_search_with_score(
    query="给教师的建议是谁写的", ranker_type="weighted", ranker_params={"k": 60}
))

for doc, score in doc_list:
    print(f"==========文档评分:{score}==========")
    print(doc.model_dump_json(indent=2))

In [ ]:
# 基于重排序模型的混合检索
from pymilvus import Function, FunctionType

# 定义生成排序器的方法
def create_cross_encoder_ranker(queries: list[str]):
    return Function(
        name="dashscope_semantic_ranker",      # ranker名词，唯一即可
        input_field_names=["text"],         # 指定原始文档字段
        function_type=FunctionType.RERANK,     # ranker类型，这里是固定值
        params={
            "reranker": "model",           # rerank类型，基于模型rerank，也就是cross-encoder
            "provider": "ali",             # rerank模型提供者
            "model_name": "qwen3-vl-rerank", # rerank模型名称
            "queries": queries,            # 查询条件
            "max_client_batch_size": 5,    # 单批次中处理的文档最大数量
        }
    )

# 2.准备问题
query = "教育的作用"

# 创建排序器
ranker = create_cross_encoder_ranker([query])

# 执行混合检索
vector_store.similarity_search_with_score(
    query=query, fetch_k=5, reranker=ranker     #每一路召回的文档数量, 与自定义交叉编码排序器
)

# 打印输出
for doc, score in doc_list:
    print(f"==========文档评分:{score}==========")
    print(doc.model_dump_json(indent=2))

In [ ]:
# 基于RRF融合混合检索的标量过滤
doc_list:list[tuple[Document, float]] = (vector_store.
    similarity_search_with_score(
        query=query,
        k=3,
        ranker_type="rrf",
        ranker_params={"k":60},
        expr="h3 like '%分化时期%'"
    ))

# 打印输出
for doc, score in doc_list:
    print(f"==========文档评分:{score}==========")
    print(doc.model_dump_json(indent=2))